In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pathlib import Path

# 设置样式（在字体设置之前）
import seaborn as sns
sns.set_style("whitegrid")
sns.set_palette("husl")

# 尝试加载本地中文字体（相对路径）
font_path = Path('fonts/SourceHanSansSC-Regular.otf')

if font_path.exists():
    # 使用本地字体
    fm.fontManager.addfont(str(font_path))
    font_prop = fm.FontProperties(fname=str(font_path))
    font_name = font_prop.get_name()
    plt.rcParams['font.sans-serif'] = [font_name]
    plt.rcParams['axes.unicode_minus'] = False
    print(f"字体设置完成: {font_name}")
else:
    print("提示: 中文字体未配置，图表中文可能显示异常")
    print(f"      请下载思源黑体到 {font_path}")
    print("      下载地址: https://github.com/adobe-fonts/source-han-sans/releases")

In [ ]:
# 黄金定投策略收益范围分析
# 本工具用于分析黄金AU9999定投策略的收益范围，帮助理解不同定投周期和起始时间对收益的影响

# 安装必要的库（如果尚未安装）
# #!pip install akshare pandas numpy matplotlib seaborn -q

import akshare as ak
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from typing import Tuple, List

print("库导入成功！")

## 1. 获取AU9999历史数据

In [ ]:
def fetch_au9999_data() -> pd.DataFrame:
    """获取上海黄金交易所AU9999历史数据"""
    print("正在获取AU9999历史数据...")
    df = ak.spot_hist_sge(symbol='Au99.99')
    
    # 转换日期格式
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    print(f"数据获取成功！共 {len(df)} 条记录")
    print(f"日期范围: {df['date'].min().date()} 至 {df['date'].max().date()}")
    print(f"最新收盘价: {df['close'].iloc[-1]:.2f} 元/克")
    
    return df

# 获取数据
df_gold = fetch_au9999_data()

# 显示最近5天数据
print("\n最近5天数据:")
df_gold.tail()

## 2. 定投策略模拟器

In [ ]:
class GoldDIPSimulator:
    """黄金定投策略模拟器（按周定投）"""
    
    def __init__(self, price_data: pd.DataFrame):
        """
        初始化模拟器
        
        参数:
            price_data: 价格数据，需包含 date 和 close 列
        """
        self.price_data = price_data.copy()
        self.price_data = self.price_data[['date', 'open', 'close', 'high', 'low']]
    
    def simulate_dip(
        self,
        start_date: str,
        end_date: str,
        weekly_investment: float = 100,
        invest_weekday: int = 0
    ) -> dict:
        """
        模拟按周定投策略
        
        参数:
            start_date: 开始日期 (YYYY-MM-DD)
            end_date: 结束日期 (YYYY-MM-DD)
            weekly_investment: 每周定投金额（元）
            invest_weekday: 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)
        
        返回:
            包含定投结果统计的字典
        """
        start = pd.to_datetime(start_date)
        end = pd.to_datetime(end_date)
        
        # 筛选日期范围
        mask = (self.price_data['date'] >= start) & (self.price_data['date'] <= end)
        period_data = self.price_data[mask].copy()
        
        if len(period_data) == 0:
            return {"error": "所选日期范围内无数据"}
        
        # 生成定投日期序列：找到第一个符合的星期几
        first_day = start
        while first_day.weekday() != invest_weekday:
            first_day += pd.Timedelta(days=1)
        
        invest_dates = pd.date_range(start=first_day, end=end, freq='W-MON') + pd.Timedelta(days=invest_weekday)
        
        # 执行定投
        total_invested = 0
        total_grams = 0
        invest_records = []
        
        for invest_date in invest_dates:
            if invest_date > end:
                break
            
            # 找到定投日的价格（T+1确认，但用定投日净值计算）
            price_data = period_data[period_data['date'] >= invest_date]
            if len(price_data) == 0:
                continue
            
            actual_date = price_data['date'].iloc[0]
            price = price_data['close'].iloc[0]
            
            # 计算购买克数
            grams = weekly_investment / price
            
            total_invested += weekly_investment
            total_grams += grams
            
            invest_records.append({
                'date': actual_date,
                'price': price,
                'amount': weekly_investment,
                'grams': grams
            })
        
        if total_grams == 0:
            return {"error": "未能执行任何定投操作"}
        
        # 计算期末价值
        final_price = period_data['close'].iloc[-1]
        final_value = total_grams * final_price
        
        # 计算收益
        profit = final_value - total_invested
        return_rate = (profit / total_invested) * 100
        
        return {
            'start_date': start_date,
            'end_date': end_date,
            'invest_count': len(invest_records),
            'total_invested': total_invested,
            'total_grams': total_grams,
            'avg_cost': total_invested / total_grams,
            'final_price': final_price,
            'final_value': final_value,
            'profit': profit,
            'return_rate': return_rate,
            'records': invest_records
        }
    
    def rolling_window_analysis(
        self,
        holding_weeks: list = None,
        weekly_investment: float = 100,
        invest_weekday: int = 0
    ) -> pd.DataFrame:
        """
        滚动窗口分析：计算所有可能的定投起始组合的收益率
        
        参数:
            holding_weeks: 持有周数列表，如 [12, 24, 48, 96, 192]
            weekly_investment: 每周定投金额
            invest_weekday: 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)
        
        返回:
            包含所有模拟结果的DataFrame
        """
        results = []
        
        for weeks in holding_weeks:
            print(f"正在分析持有 {weeks} 周的定投策略...")
            
            # 计算滚动窗口
            for i in range(len(self.price_data) - weeks):
                start_row = self.price_data.iloc[i]
                end_idx = i + weeks
                
                if end_idx >= len(self.price_data):
                    continue
                
                end_row = self.price_data.iloc[end_idx]
                
                start_date = start_row['date']
                end_date = end_row['date']
                
                # 计算定投收益
                result = self.simulate_dip(
                    start_date=start_date.strftime('%Y-%m-%d'),
                    end_date=end_date.strftime('%Y-%m-%d'),
                    weekly_investment=weekly_investment,
                    invest_weekday=invest_weekday
                )
                
                if 'error' not in result:
                    results.append({
                        'holding_weeks': weeks,
                        'start_date': start_date,
                        'end_date': end_date,
                        'total_invested': result['total_invested'],
                        'final_value': result['final_value'],
                        'profit': result['profit'],
                        'return_rate': result['return_rate'],
                        'avg_cost': result['avg_cost'],
                        'final_price': result['final_price']
                    })
        
        return pd.DataFrame(results)

print("定投模拟器类已定义！（按周定投）")

## 3. 可视化函数

In [ ]:
def plot_return_distribution(results_df: pd.DataFrame, holding_weeks: int):
    """绘制特定持有期收益率分布"""
    data = results_df[results_df['holding_weeks'] == holding_weeks]
    
    if len(data) == 0:
        print(f"没有持有期为 {holding_weeks} 周的数据")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'黄金定投 {holding_weeks} 周收益率分布分析', fontsize=16, fontweight='bold')
    
    # 1. 收益率直方图
    ax1 = axes[0, 0]
    ax1.hist(data['return_rate'], bins=50, edgecolor='black', alpha=0.7, color='steelblue')
    ax1.axvline(data['return_rate'].mean(), color='red', linestyle='--', linewidth=2, label=f"平均: {data['return_rate'].mean():.2f}%")
    ax1.axvline(0, color='green', linestyle='-', linewidth=1, alpha=0.5)
    ax1.set_xlabel('收益率 (%)')
    ax1.set_ylabel('频数')
    ax1.set_title('收益率分布')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. 盈亏比例
    ax2 = axes[0, 1]
    profit_count = (data['return_rate'] > 0).sum()
    loss_count = (data['return_rate'] <= 0).sum()
    colors = ['#2ecc71' if profit_count > loss_count else '#e74c3c']
    ax2.pie([profit_count, loss_count], labels=[f'盈利\n{profit_count}次', f'亏损\n{loss_count}次'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
    ax2.set_title(f'盈利概率: {profit_count/len(data)*100:.1f}%')
    
    # 3. 收益率时间序列
    ax3 = axes[1, 0]
    ax3.plot(data['start_date'], data['return_rate'], marker='o', markersize=2, linewidth=1)
    ax3.axhline(0, color='green', linestyle='-', linewidth=1, alpha=0.5)
    ax3.fill_between(data['start_date'], data['return_rate'], 0, 
                     where=(data['return_rate'] >= 0), color='#2ecc71', alpha=0.3, label='盈利')
    ax3.fill_between(data['start_date'], data['return_rate'], 0, 
                     where=(data['return_rate'] < 0), color='#e74c3c', alpha=0.3, label='亏损')
    ax3.set_xlabel('开始定投日期')
    ax3.set_ylabel('收益率 (%)')
    ax3.set_title('收益率随时间变化')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)
    
    # 4. 统计指标
    ax4 = axes[1, 1]
    ax4.axis('off')
    stats_text = f"""
【统计指标】
━━━━━━━━━━━━━━━━━━━━━━
样本数量:      {len(data)}
平均收益率:   {data['return_rate'].mean():+.2f}%
中位数收益率: {data['return_rate'].median():+.2f}%
最大收益:     {data['return_rate'].max():+.2f}%
最大亏损:     {data['return_rate'].min():+.2f}%
标准差:       {data['return_rate'].std():.2f}%
盈利次数:     {profit_count} ({profit_count/len(data)*100:.1f}%)
亏损次数:     {loss_count} ({loss_count/len(data)*100:.1f}%)
    """
    ax4.text(0.1, 0.5, stats_text, fontsize=12,
             verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

def plot_holding_period_comparison(results_df: pd.DataFrame):
    """比较不同持有期的收益率"""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('不同持有期收益率对比', fontsize=16, fontweight='bold')
    
    # 按持有期分组统计
    summary = results_df.groupby('holding_weeks')['return_rate'].agg([
        ('mean', 'mean'),
        ('median', 'median'),
        ('min', 'min'),
        ('max', 'max'),
        ('std', 'std'),
        ('count', 'count'),
        ('profit_ratio', lambda x: (x > 0).sum() / len(x) * 100)
    ]).reset_index()
    
    weeks = summary['holding_weeks'].values
    x = np.arange(len(weeks))
    width = 0.35
    
    # 1. 平均收益率和中位数
    ax1 = axes[0, 0]
    ax1.bar(x - width/2, summary['mean'], width, label='平均', color='steelblue', alpha=0.8)
    ax1.bar(x + width/2, summary['median'], width, label='中位数', color='coral', alpha=0.8)
    ax1.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax1.set_xlabel('持有周数')
    ax1.set_ylabel('收益率 (%)')
    ax1.set_title('平均收益率 vs 中位数')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'{w}周' for w in weeks])
    ax1.legend()
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 2. 收益率范围
    ax2 = axes[0, 1]
    ax2.fill_between(x, summary['min'], summary['max'], alpha=0.3, color='gray', label='收益范围')
    ax2.plot(x, summary['mean'], marker='o', linewidth=2, label='平均收益', color='red')
    ax2.axhline(0, color='black', linestyle='-', linewidth=0.8)
    ax2.set_xlabel('持有周数')
    ax2.set_ylabel('收益率 (%)')
    ax2.set_title('收益率范围')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'{w}周' for w in weeks])
    
    # 3. 盈利概率
    ax3 = axes[1, 0]
    colors = ['#2ecc71' if x >= 50 else '#e67e22' if x >= 40 else '#e74c3c' for x in summary['profit_ratio']]
    ax3.bar(x, summary['profit_ratio'], color=colors, alpha=0.8)
    ax3.axhline(50, color='black', linestyle='--', linewidth=1, alpha=0.5, label='50%')
    ax3.set_xlabel('持有周数')
    ax3.set_ylabel('盈利概率 (%)')
    ax3.set_title('定投盈利概率')
    ax3.set_xticks(x)
    ax3.set_xticklabels([f'{w}周' for w in weeks])
    ax3.set_ylim([0, 100])
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. 统计表格
    ax4 = axes[1, 1]
    ax4.axis('off')
    table_data = []
    for _, row in summary.iterrows():
        table_data.append([
            f"{row['holding_weeks']}周",
            f"{row['mean']:+.1f}%",
            f"{row['profit_ratio']:.1f}%",
            f"{row['min']:+.1f}% ~ {row['max']:+.1f}%"
        ])
    
    table = ax4.table(cellText=table_data, 
                     colLabels=['持有期', '平均收益', '盈利概率', '收益范围'],
                     cellLoc='center',
                     loc='center')
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2)
    # 表头
    for i in range(4):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    # 数据行
    for i in range(1, len(table_data) + 1):
        for j in range(4):
            if i % 2 == 0:
                table[(i, j)].set_facecolor('#E7E6E6')
    
    ax4.set_title('统计汇总', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    return summary

print("可视化函数已定义！")

In [ ]:
# 设置分析日期和参数
start_date = '2022-02-24'  # 开始日期
end_date = '2025-12-31'    # 结束日期
weekly_investment = 100    # 每周定投金额（元）
invest_weekday = 0         # 每周星期几定投 (0=周一, 1=周二, ..., 6=周日)

# 基于日期范围自动计算持有期（周数）
start = pd.to_datetime(start_date)
end = pd.to_datetime(end_date)
total_weeks = (end - start).days // 7
# 生成从12周到总周数的持有期列表（间隔12周，约3个月）
holding_weeks = list(range(12, total_weeks + 1, 12))

print(f"分析日期范围: {start_date} 至 {end_date}")
print(f"总周数: {total_weeks} 周")
print(f"持有期分析: {holding_weeks} 周")

In [ ]:
# 创建模拟器
simulator = GoldDIPSimulator(df_gold)

print("模拟器已创建！")
print(f"可分析的数据范围: {df_gold['date'].min().date()} 至 {df_gold['date'].max().date()}")

### 4.1 单次定投模拟示例

In [ ]:
# 使用设置参数运行单次模拟
result = simulator.simulate_dip(
    start_date=start_date,
    end_date=end_date,
    weekly_investment=weekly_investment,
    invest_weekday=invest_weekday
)

print("定投结果:")
print(f"  开始日期: {result['start_date']}")
print(f"  结束日期: {result['end_date']}")
print(f"  定投次数: {result['invest_count']} 次")
print(f"  总投入: {result['total_invested']:.2f} 元")
print(f"  累计黄金: {result['total_grams']:.2f} 克")
print(f"  平均成本: {result['avg_cost']:.2f} 元/克")
print(f"  期末价格: {result['final_price']:.2f} 元/克")
print(f"  期末价值: {result['final_value']:.2f} 元")
print(f"  总收益: {result['profit']:+.2f} 元")
print(f"  收益率: {result['return_rate']:+.2f}%")

### 4.2 滚动窗口分析（可调整参数）

In [ ]:
# 运行滚动窗口分析（使用自动计算的持有期）
results_df = simulator.rolling_window_analysis(
    holding_weeks=holding_weeks,
    weekly_investment=weekly_investment,
    invest_weekday=invest_weekday
)

print(f"\n分析完成！共生成 {len(results_df)} 个样本")
results_df.head()

### 4.3 可视化结果

In [ ]:
# 显示中间持有期的详细分析（自动选择）
selected_weeks = holding_weeks[len(holding_weeks) // 2]  # 选择中间的持有期

if selected_weeks in results_df['holding_weeks'].values:
    plot_return_distribution(results_df, selected_weeks)
else:
    print(f"警告：{selected_weeks}周不在分析结果中")

In [ ]:
# 不同持有期对比
summary = plot_holding_period_comparison(results_df)

## 5. 额外测试

在这里修改参数进行额外的测试分析：

```python
# 修改下面的参数进行测试
custom_result = simulator.simulate_dip(
    start_date='2020-01-01',   # 修改测试开始日期
    end_date='2025-12-31',     # 修改测试结束日期
    weekly_investment=200,     # 修改每周定投金额
    invest_weekday=0           # 修改定投星期几 (0=周一, ..., 6=周日)
)

print("测试定投结果:")
for key, value in custom_result.items():
    if key != 'records':
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")
```

In [ ]:
# 复制上面的代码到此处运行测试

custom_result = simulator.simulate_dip(
    start_date='2020-01-01',
    end_date='2025-12-31',
    weekly_investment=200,
    invest_weekday=0
)

print("测试定投结果:")
for key, value in custom_result.items():
    if key != 'records':
        if isinstance(value, float):
            print(f"  {key}: {value:.2f}")
        else:
            print(f"  {key}: {value}")